# Capital Allocation I

The ideas are in the slides. Here you build the frontier one step at a time, on the six factors.

**The plan**

1. The six factors
2. The tangency portfolio, line by line
3. 🎯 The minimum-variance portfolio: your turn
4. Two portfolios draw the frontier
5. 🎯 One portfolio, many goals: your turn
6. 🎯 Does a new factor move the frontier? The alpha test. Then back to the slides
7. 🎯 Challenge (homework): goals and mandates on the 49 industries

📑 **Slides for this class:** [Capital Allocation I (PDF)](https://amoreira2.github.io/UG54/slides/capital_allocation_1.pdf)

<iframe src="../../slides/capital_allocation_1.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

ff = pd.read_csv(f"{BASE}/ff_monthly.csv", index_col=0, parse_dates=True)   # Ken French, monthly
F  = ff[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'UMD']].dropna()             # the six factors, excess returns

ind = pd.read_parquet(f"{BASE}/ind49_monthly_1926.parquet")                 # 49 industries, for the challenge
fac = pd.read_parquet(f"{BASE}/factors_monthly_1926.parquet")
X   = ind.sub(fac['RF'], axis=0).loc['1980-01':'2026-06']                   # their excess returns

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)

print(f"six factors {F.index[0]:%Y-%m} to {F.index[-1]:%Y-%m}, {len(F)} months; {X.shape[1]} industries for the challenge")

---

## 1. The six factors

`F` has one column per factor, one row per month. They are excess returns: what you earn over T-bills. Start by looking at
them on their own.

In [ ]:
summary = pd.DataFrame({'mean': F.mean() * 12, 'volatility': F.std() * np.sqrt(12), 'Sharpe': F.apply(sharpe)})
print(summary.round(3).to_string(), "\n")
print("correlations")
print(F.corr().round(2).to_string())

- Which factor has the highest Sharpe ratio on its own? Hold on to that number.
- Look for a large negative correlation. Why would the optimizer like it?

---

## 2. The tangency portfolio, line by line

The formula from the slides is $W^{\star} = \frac{1}{\gamma}\Sigma^{-1}\mu$. It needs two inputs.

**Step 1: the inputs.** Everything per year.

In [ ]:
mu = F.mean() * 12      # expected excess return of each factor
S  = F.cov() * 12       # covariance matrix

print(mu.round(3).to_string())

**Step 2: the formula, with $\gamma = 1$.** `np.linalg.solve(S, mu)` computes $\Sigma^{-1}\mu$. It gives the same answer as
`np.linalg.inv(S) @ mu`, and it is more accurate.

In [ ]:
W = pd.Series(np.linalg.solve(S, mu), index=F.columns)

print(W.round(2).to_string())
print(f"\nthese add to {W.sum():.1f}")

These weights are large. Hold on to that as well: we come back to it in section 5.

**Step 3: set the scale.** The mix is what matters, so divide by the total to make the weights add to one. Dividing every
weight by the same number changes the size of the portfolio, not what is in it. This is the tangency portfolio, `T`.

A factor is already a long-short position, so here "adds to one" is not a budget. It is just a convenient scale.

In [ ]:
T = W / W.sum()

print(T.round(3).to_string())

**Step 4: measure it.** There are two ways to get a portfolio's mean and volatility: from the weights and the inputs, or by
building its return in each month. They must agree. The Sharpe ratio has a third way, $\sqrt{\mu'\Sigma^{-1}\mu}$.

In [ ]:
# from the weights and the inputs
mean_T = T @ mu
vol_T  = np.sqrt(T @ S @ T)

# from its return in each month
r_T = F @ T

print(f"mean        {mean_T:.2%}   from the monthly returns {r_T.mean() * 12:.2%}")
print(f"volatility  {vol_T:.2%}   from the monthly returns {r_T.std() * np.sqrt(12):.2%}")
print(f"Sharpe      {mean_T / vol_T:.3f}   from the monthly returns {sharpe(r_T):.3f}   from the formula {np.sqrt(mu @ W):.3f}")

- Compare with the best single factor in section 1. Where did the extra Sharpe ratio come from?
- Which factors get the most weight? Are they the ones with the highest Sharpe ratios?
- The volatility is lower than that of *any* factor. How is that possible?

---

## 3. 🎯 The minimum-variance portfolio

Same steps, with one change. Its formula is $\Sigma^{-1}\mathbf{1}$: where the tangency portfolio had $\mu$, there is now a
vector of ones.

1. Compute $\Sigma^{-1}\mathbf{1}$. A vector of six ones is `np.ones(6)`. Copy Step 2.
2. Divide by its sum, so the weights add to one. Copy Step 3.

Call the result `w_mv`.

In [ ]:
# === YOUR TURN ===

w_mv = ____

In [ ]:
#@title 🔒 Check: minimum variance
MV = pd.Series(np.linalg.solve(S, np.ones(6)), index=F.columns)
MV = MV / MV.sum()
mean_MV, vol_MV = MV @ mu, np.sqrt(MV @ S @ MV)

try:
    print("yours matches" if np.allclose(np.asarray(w_mv, dtype=float).ravel(), MV.values) else "yours differs: compare with the table below", "\n")
except Exception as e:
    print("yours: not built yet:", type(e).__name__, "\n")

print(pd.DataFrame({'tangency': T, 'minimum variance': MV}).round(3).to_string(), "\n")
print(f"tangency          mean {mean_T:.2%}, volatility {vol_T:.2%}, Sharpe {mean_T / vol_T:.2f}")
print(f"minimum variance  mean {mean_MV:.2%}, volatility {vol_MV:.2%}, Sharpe {mean_MV / vol_MV:.2f}")

- Minimum variance never uses $\mu$. So what would it do if you doubled every factor's expected return?
- Which one has the lower volatility? Which has the higher Sharpe ratio? Could it be otherwise?

---

## 4. Two portfolios draw the frontier

Every portfolio on the curve is a mix of these two: $a \times$ tangency $+ (1-a) \times$ minimum variance. So we can draw the
curve with a loop. Each pass picks an $a$, builds the weights, and records the volatility and the mean.

With T-bills, you can reach any point on the straight line through the tangency portfolio.

In [ ]:
curve = []
for a in np.linspace(-12, 30, 400):
    w = a * T + (1 - a) * MV
    curve.append((np.sqrt(w @ S @ w), w @ mu))
curve = np.array(curve)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curve[:, 0], curve[:, 1], color='grey', label='the curve: mixes of the two')
ax.plot([0, 0.18], [0, 0.18 * mean_T / vol_T], color='firebrick', label=f'the line: slope {mean_T / vol_T:.2f}')
ax.scatter(summary['volatility'], summary['mean'], color='black', s=20)
for c in F:
    ax.annotate(c, (summary.loc[c, 'volatility'], summary.loc[c, 'mean']), fontsize=9)
ax.scatter(vol_T, mean_T, marker='*', s=220, color='firebrick', label='tangency', zorder=5)
ax.scatter(vol_MV, mean_MV, color='tab:blue', s=60, label='minimum variance', zorder=5)
ax.set_xlim(0, 0.18); ax.set_ylim(0, 0.14); ax.set_xlabel('volatility'); ax.set_ylabel('expected return over T-bills'); ax.legend()
plt.show()

- Which $a$ gives the tangency portfolio? Which gives minimum variance?
- Where are the single factors, relative to the curve? What does the distance mean?
- What does a point on the line to the right of the star mean, in terms of what you hold?

---

## 5. 🎯 One portfolio, many goals

Everyone on the line holds the same mix, `T`, in a different size: $x$ times `T`, with the rest in T-bills. Then

$$\text{mean} = x \times \text{mean}_T \qquad\qquad \text{volatility} = x \times \text{vol}_T$$

So a goal is just a choice of $x$. Risk aversion is the one we can read off `W`, since $W/\gamma$ is `W.sum()` $/\gamma$ times `T`:

In [ ]:
x_gamma1 = W.sum() / 1
x_gamma5 = W.sum() / 5

print(f"gamma = 1: x = {x_gamma1:.1f}, volatility {x_gamma1 * vol_T:.0%}")
print(f"gamma = 5: x = {x_gamma5:.1f}, volatility {x_gamma5 * vol_T:.0%}")

Now two goals a client might actually bring. Use `vol_T` and `mean_T` from Step 4, and the two equations above.

- `x_vol`: the client wants **10% volatility**
- `x_ret`: the client wants **8% a year over T-bills**

In [ ]:
# === YOUR TURN ===

x_vol = ____
x_ret = ____

In [ ]:
#@title 🔒 Check: four goals, one line
goals = {'gamma = 1': x_gamma1, 'gamma = 5': x_gamma5, '10% volatility': 0.10 / vol_T, '8% return': 0.08 / mean_T}

table = pd.DataFrame({'x': goals}).assign(mean=lambda d: d.x * mean_T, volatility=lambda d: d.x * vol_T)
table['Sharpe'] = table['mean'] / table['volatility']
print(table.round(3).to_string(), "\n")

try:
    print(f"yours: x_vol {x_vol:.2f}, x_ret {x_ret:.2f}", "\n")
except Exception as e:
    print("yours: not built yet:", type(e).__name__, "\n")

print("the weights, for the 10% volatility client")
print((goals['10% volatility'] * T).round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curve[:, 0], curve[:, 1], color='grey')
ax.plot([0, 0.26], [0, 0.26 * mean_T / vol_T], color='firebrick')
ax.scatter(summary['volatility'], summary['mean'], color='black', s=20)
for c in F:
    ax.annotate(c, (summary.loc[c, 'volatility'], summary.loc[c, 'mean']), fontsize=9)
ax.scatter(vol_T, mean_T, marker='*', s=220, color='firebrick', zorder=5)
ax.annotate('tangency', (vol_T, mean_T), xytext=(-55, 8), textcoords='offset points')
ax.scatter(vol_MV, mean_MV, color='tab:blue', s=60, zorder=5)
for name in ['gamma = 5', '10% volatility', '8% return']:
    ax.scatter(table.loc[name, 'volatility'], table.loc[name, 'mean'], color='firebrick', s=60, zorder=5)
    ax.annotate(name, (table.loc[name, 'volatility'], table.loc[name, 'mean']), xytext=(-70, 8), textcoords='offset points')
ax.set_xlim(0, 0.26); ax.set_ylim(0, 0.30); ax.set_xlabel('volatility'); ax.set_ylabel('expected return over T-bills')
plt.show()

- `gamma = 1` is not on the picture. Where would it be? What would you tell a client who gave you that number?
- All four have the same Sharpe ratio. Why?
- Which part of the table changes if the client changes their mind, and which never does?

---

## 6. 🎯 Does a new factor move the frontier?

You have seen this regression all semester. Here is what it has been testing.

Suppose you hold five factors and someone offers you momentum. Does it improve your best portfolio? The answer is one
regression away: put it on the **tangency portfolio of what you already hold**.

**Step 1: the tangency portfolio of the other five.** The same two lines as before.

In [ ]:
others = [c for c in F if c != 'UMD']

W5 = np.linalg.solve(F[others].cov(), F[others].mean())
T5 = F[others] @ (W5 / W5.sum())                        # its return in each month
T5.name = 'tangency of the other five'

**Step 2: the alpha test, two ways.** On that one portfolio, then on all five factors. Look at the alpha in each.

In [ ]:
on_tangency = sm.OLS(F['UMD'], sm.add_constant(T5)).fit()
on_all_five = sm.OLS(F['UMD'], sm.add_constant(F[others])).fit()

print(f"on the tangency portfolio:  alpha {on_tangency.params['const'] * 12:.2%} a year, t = {on_tangency.tvalues['const']:.2f}")
print(f"on all five factors:        alpha {on_all_five.params['const'] * 12:.2%} a year, t = {on_all_five.tvalues['const']:.2f}")

Same alpha. A regression on five factors and a regression on their tangency portfolio ask the same question: *can what you
already hold reproduce this asset's return?* An alpha of zero means yes. Then the new asset is **spanned**, and the frontier
does not move.

**Step 3: how far it moves.** The appraisal ratio, $\alpha/\sigma_\varepsilon$, adds to the Sharpe ratio in squares. Use the residual from
the regression on all five, which is the best hedge those five allow.

In [ ]:
sr_without = sharpe(T5)
appraisal  = on_all_five.params['const'] / on_all_five.resid.std() * np.sqrt(12)
sr_with    = sharpe(r_T)

print(f"best Sharpe without momentum    {sr_without:.3f}")
print(f"appraisal ratio                 {appraisal:.3f}")
print(f"sqrt(without^2 + appraisal^2)   {np.sqrt(sr_without ** 2 + appraisal ** 2):.3f}")
print(f"best Sharpe with momentum       {sr_with:.3f}")

**Your turn: HML.** Copy the regression on all five, change `'UMD'` to `'HML'`, and leave HML out of the right-hand side. Call
its alpha `alpha_hml`, in decimals a year.

In [ ]:
# === YOUR TURN ===

alpha_hml = ____

In [ ]:
#@title 🔒 Check: every factor, against the other five
rows = {}
for c in F:
    rest = [o for o in F if o != c]
    fit  = sm.OLS(F[c], sm.add_constant(F[rest])).fit()
    w5   = np.linalg.solve(F[rest].cov(), F[rest].mean())
    rows[c] = {'Sharpe alone': sharpe(F[c]), 'alpha': fit.params['const'] * 12, 'se of alpha': fit.bse['const'] * 12, 't': fit.tvalues['const'],
               'best without': sharpe(F[rest] @ w5), 'best with': sharpe(r_T)}
print(pd.DataFrame(rows).T.round(3).to_string(), "\n")

try:
    print(f"yours: alpha_hml {alpha_hml:.2%}")
except Exception as e:
    print("yours: not built yet:", type(e).__name__)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
for ax, c in zip(axes, ['HML', 'UMD']):
    rest = [o for o in F if o != c]
    m5, S5 = F[rest].mean() * 12, F[rest].cov() * 12
    t5, g5 = np.linalg.solve(S5, m5), np.linalg.solve(S5, np.ones(5))
    t5, g5 = t5 / t5.sum(), g5 / g5.sum()
    cv = np.array([(np.sqrt(w @ S5 @ w), w @ m5) for w in [a * t5 + (1 - a) * g5 for a in np.linspace(-12, 30, 400)]])
    ax.plot(cv[:, 0], cv[:, 1], color='silver', lw=4, label='without it')
    ax.plot(curve[:, 0], curve[:, 1], color='firebrick', lw=1.3, label='with it')
    ax.plot([0, 0.16], [0, 0.16 * (t5 @ m5) / np.sqrt(t5 @ S5 @ t5)], color='silver', lw=4)
    ax.plot([0, 0.16], [0, 0.16 * mean_T / vol_T], color='firebrick', lw=1.3)
    ax.scatter(summary.loc[c, 'volatility'], summary.loc[c, 'mean'], color='black', zorder=5)
    ax.set_title(f"{c}: alpha {rows[c]['alpha']:.1%}, t = {rows[c]['t']:.1f}"); ax.set_xlabel('volatility')
    ax.set_xlim(0, 0.16); ax.set_ylim(0, 0.14)
axes[0].set_ylabel('expected return over T-bills'); axes[0].legend(loc='upper left')
plt.show()

- HML has a positive Sharpe ratio on its own, yet it adds nothing. How can both be true?
- The market has the largest alpha, larger than its own mean. Why would it be bigger than its mean?
- SMB's Sharpe ratio is about zero, and so is what it adds. Is that always how it goes? Look at HML before you answer.
- Alpha of HML: read the standard error too. Would you drop value from a real portfolio on this evidence?
  **Bring this back to the slides.**

---

## ⏸ Back to the slides

The challenge below is homework. It uses the second half of the slides, and section 5 of this notebook.

---

## 🎯 Challenge: goals and mandates

*Homework — due before the next class.*

From here on, use `X`: the 49 industries, in excess returns, 1980–2026. Allow short positions unless told otherwise.

### Q1 — The tangency portfolio

Build the tangency portfolio on the 49 industries. Report its Sharpe ratio. How many short positions does it have, and
what is its gross exposure when it runs 10% volatility? (Gross exposure: the absolute weights added up.)

> **📌 Required variable names:**
> ```python
> tan_sharpe = ____   # in-sample Sharpe ratio of the 49-industry tangency portfolio
> ```

In [ ]:
# Your work here


tan_sharpe = ____

print(f"tangency Sharpe: {tan_sharpe:.3f}")

### Q2 — Three goals

A client comes with a goal. Every portfolio below uses the tangency mix, plus T-bills, unless told otherwise.

**(a) A volatility goal.** The client wants 10% volatility. How much of the money sits in T-bills?

**(b) A return goal.** The client wants 8% a year over T-bills. What volatility must they accept?

**(c) A leverage limit.** The client's broker caps gross exposure at 2. Scale the tangency mix until it hits the cap. What
volatility do they get? Can they reach the 10% goal in (a)?

**(d) Both.** Now they insist on 10% volatility **and** gross exposure of at most 2, so the mix has to change. Find the
portfolio with the highest expected return under both limits, and report its Sharpe ratio. This takes a numerical
optimizer: use AI, and check both limits hold in the answer.

> **📌 Required variable names:**
> ```python
> tbills_at_10vol = ____   # share of the money in T-bills in (a), a decimal
> vol_for_8pct    = ____   # volatility needed in (b), a decimal
> vol_at_gross2   = ____   # volatility at the cap in (c), a decimal
> sr_gross2       = ____   # Sharpe ratio of the best portfolio in (d)
> ```

In [ ]:
# Your work here


tbills_at_10vol = ____
vol_for_8pct    = ____
vol_at_gross2   = ____
sr_gross2       = ____

print(tbills_at_10vol, vol_for_8pct, vol_at_gross2, sr_gross2)

### Q3 — What a mandate costs, as it widens

Your client's exclusion list keeps growing. At each stage, drop those industries, rebuild the tangency portfolio on what
is left, and report its Sharpe ratio.

| stage | excluded, added to everything before |
|---|---|
| 1 | `Smoke` |
| 2 | `Guns`, `Beer` |
| 3 | `Coal`, `Oil` |
| 4 | `Mines`, `Aero`, `Fun` |
| 5 | `Soda`, `Chems`, `Steel`, `Autos`, `Trans`, `Util` |

Plot the Sharpe ratio by stage. Then turn it into money: at 10% volatility, how much expected return a year does the
client give up at each stage?

**A control.** Dropping *any* asset lowers an in-sample maximum, so some of this cost has nothing to do with sin. At stage 5,
14 industries are gone. Drop 14 industries **at random**, 300 times, using `rng = np.random.default_rng(54)`, and report the
average Sharpe ratio. How much of the stage 5 cost is about these industries, and how much is just having fewer assets?

> **📌 Required variable names:**
> ```python
> sr_stage1 = ____    # Sharpe ratio after stage 1
> sr_stage2 = ____
> sr_stage3 = ____
> sr_stage4 = ____
> sr_stage5 = ____
> sr_random14 = ____  # average Sharpe ratio after dropping 14 at random
> ```

In [ ]:
# Your work here


sr_stage1 = ____
sr_stage2 = ____
sr_stage3 = ____
sr_stage4 = ____
sr_stage5 = ____
sr_random14 = ____

print(sr_stage1, sr_stage2, sr_stage3, sr_stage4, sr_stage5, sr_random14)

### Q4 — The memo

> **📝 Your task — maximum ten sentences.**
>
> Your client is an endowment considering stage 3. Say what it costs, in Sharpe ratio and in return a year, and how much you
> trust that number given the control. Then the harder question: **suppose a large share of all investors adopted the same
> list. What happens to the prices and the expected returns of those industries?** Who is harmed and who benefits, over the
> transition and afterwards: the endowment, the investors who keep holding, the excluded firms, the firms that remain?
> Finish with what this implies about the cost you measured, and about whether exclusion changes what those firms do.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["tan_sharpe", "tbills_at_10vol", "vol_for_8pct", "vol_at_gross2", "sr_gross2", "sr_stage1", "sr_stage2", "sr_stage3", "sr_stage4", "sr_stage5", "sr_random14", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L12_CapitalAllocationI_v2_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")